<a href="https://colab.research.google.com/github/KoyuChen/Horton_Manning_Replication/blob/main/Replication_Studies_Full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-1.7B-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen3-1.7B-unsloth-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.3.3: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.41G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-1.7B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151669)
    (layers): ModuleList(
      (0-1): 2 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear4bit(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
      

In [ ]:
# ============================================================
# General Social Agents (Section 3) - Paper-Style Toy Replication
#   11–20 Money Request Game (basic + costless + cycle)
#   - Selection objective: CDF-L1 distance
#   - Two-stage elicitation (thoughts -> final choice)
#   - Invalid discard (no resampling)
#   - Population simulation (sample N agents by weights; each plays once)
#   - Negative controls: MBTI / Historical figures / Always-pick-N
#
# REQUIREMENTS:
#   - You already have `model, tokenizer` loaded (Unsloth or HF).
#   - For Unsloth: FastLanguageModel.for_inference(model)
#
# Outputs:
#   - weights learned on BASIC
#   - validation KL on COSTLESS and CYCLE vs baseline
#   - negative control performance + weight collapse behaviors
# ============================================================

import re
import math
import numpy as np
import random
import torch
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

# If using Unsloth:
# from unsloth import FastLanguageModel
# FastLanguageModel.for_inference(model)

# -----------------------------
# 0) Reproducibility helpers
# -----------------------------
def set_global_seed(seed: int, deterministic: bool = False) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def derive_seed(master_seed: int, stage: int, game_i: int, agent_i: int, persona_k: int) -> int:
    # stage: 0 fit/basic, 1 val/costless, 2 val/cycle, 3 baseline
    return int(master_seed + 1_000_000*stage + 10_000*game_i + 100*agent_i + persona_k)

# -----------------------------
# 1) Game definitions
# -----------------------------
@dataclass
class MoneyRequestGame:
    variant: str  # "basic" | "costless" | "cycle"
    text: str

def build_1120_game_text(variant: str) -> str:
    """
    Natural-language description of the 11–20 money request game variants.
    Keep this neutral and mechanical.
    """
    if variant == "basic":
        return (
            "You and another player will each request an integer amount from 11 to 20 dollars.\n"
            "Both players receive the amount they request.\n"
            "Additionally, if you request exactly 1 dollar less than the other player, you receive a bonus of 20 dollars.\n"
            "For example: if you request 19 and the other requests 20, you get 19 + 20 = 39.\n"
            "Your goal is to maximize your own payoff."
        )
    if variant == "costless":
        # toy text consistent with "costless undercutting" idea:
        return (
            "You and another player will each request an integer amount from 11 to 20 dollars.\n"
            "Both players receive the amount they request, BUT requesting less than 20 has a reduced opportunity cost:\n"
            "If you request any amount from 11 to 19, you still receive 17 dollars as a base (instead of your request).\n"
            "If you request 20, you receive 20 dollars.\n"
            "Additionally, if you request exactly 1 dollar less than the other player, you receive a bonus of 20 dollars.\n"
            "Your goal is to maximize your own payoff."
        )
    if variant == "cycle":
        # toy text consistent with cycle bonus (wrap-around):
        return (
            "You and another player will each request an integer amount from 11 to 20 dollars.\n"
            "Both players receive the amount they request.\n"
            "Additionally, if you request exactly 1 dollar less than the other player, you receive a bonus of 20 dollars.\n"
            "In this variant there is also a wrap-around bonus:\n"
            "If the other player requests 11 and you request 20, you also receive the bonus of 20 dollars.\n"
            "Your goal is to maximize your own payoff."
        )
    raise ValueError("variant must be basic/costless/cycle")

GAMES = [
    MoneyRequestGame("basic", build_1120_game_text("basic")),
    MoneyRequestGame("costless", build_1120_game_text("costless")),
    MoneyRequestGame("cycle", build_1120_game_text("cycle")),
]

ACTION_SET = list(range(11, 21))  # 11..20

# -----------------------------
# 2) Targets (human distributions) - Real data from Arad & Rubinstein (2012) Table 1
# -----------------------------
def real_human_dist_basic() -> Dict[int, float]:
    # Basic (n=108), percentages rounded as in paper Table 1
    # Sum to ~100%, normalized
    p = {11: 0.04, 12: 0.00, 13: 0.03, 14: 0.06, 15: 0.01, 16: 0.06,
         17: 0.32, 18: 0.30, 19: 0.12, 20: 0.06}
    s = sum(p.values())
    return {k: v / s for k, v in p.items()}

def real_human_dist_costless() -> Dict[int, float]:
    # Costless (n=53)
    p = {11: 0.00, 12: 0.04, 13: 0.00, 14: 0.04, 15: 0.04, 16: 0.04,
         17: 0.09, 18: 0.21, 19: 0.40, 20: 0.15}
    s = sum(p.values())
    return {k: v / s for k, v in p.items()}

def real_human_dist_cycle() -> Dict[int, float]:
    # Cycle (n=72)
    p = {11: 0.01, 12: 0.01, 13: 0.00, 14: 0.01, 15: 0.00, 16: 0.04,
         17: 0.10, 18: 0.22, 19: 0.47, 20: 0.13}
    s = sum(p.values())
    return {k: v / s for k, v in p.items()}

HUMAN_DIST = {
    "basic": real_human_dist_basic(),
    "costless": real_human_dist_costless(),
    "cycle": real_human_dist_cycle(),
}

# -----------------------------
# 3) Prompt families
# -----------------------------
SYSTEM_PROMPT = "You are a careful decision-maker."

STRATEGIC_PROMPTS: List[Tuple[str, str]] = [
    ("level0_anchor20", "You do not think strategically. Choose the most obvious/highest number."),
    ("level1_undercut", "Assume the other player chooses 20. Choose the best response."),
    ("level2_undercut", "Assume the other player best-responds to 20 by choosing 19. Choose the best response."),
    ("level3_undercut", "Assume the other player best-responds to 19 by choosing 18. Choose the best response."),
    ("level4_undercut", "Assume the other player best-responds to 18 by choosing 17. Choose the best response."),
    ("level5_undercut", "Assume the other player best-responds to 17 by choosing 16. Choose the best response."),
    ("level_k_var13", "You vary between a 1-level, 2-level, and 3-level thinker."),
    ("level_k_var23", "You think approximately 2 to 3 levels ahead."),
    ("level0_random", "You are a level-0 thinker but choose with some randomness."),
    ("mixed_reasoning_1to3", "You use limited strategic reasoning, roughly between level-1 and level-3."),
]

# Negative controls
MBTI_TYPES = [
    "ISTJ","ISFJ","INFJ","INTJ",
    "ISTP","ISFP","INFP","INTP",
    "ESTP","ESFP","ENFP","ENTP",
    "ESTJ","ESFJ","ENFJ","ENTJ",
]
MBTI_PROMPTS: List[Tuple[str, str]] = [(t, f"You have personality type {t}. Decide accordingly.") for t in MBTI_TYPES]

HISTORICAL_FIGURES = [
    "Julius Caesar","Confucius","Napoleon","Gandhi","Cleopatra","Aristotle",
    "Queen Victoria","Alexander the Great","Sun Tzu","Einstein","Churchill",
    "Joan of Arc","Shakespeare","Lincoln","Marie Curie","Mandela","Socrates","Newton",
]
HIST_PROMPTS: List[Tuple[str, str]] = [(h, f"You are {h}. Decide as {h} would.") for h in HISTORICAL_FIGURES]

ALWAYS_PICK_PROMPTS: List[Tuple[str, str]] = [(f"always_{n}", f"You always request {n}. No matter what.") for n in ACTION_SET]

BASELINE_PROMPT = ("baseline", "You are a typical human participant. Choose what you would request.")

# -----------------------------
# 4) Two-stage elicitation + parsing
# -----------------------------
def apply_chat_template(messages):
    enc = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,        # 关键：要 dict 才有 attention_mask
    )
    return {k: v.to(model.device) for k, v in enc.items()}

def _generate_text(enc, max_new_tokens=128, temperature=1.0, top_p=0.95, top_k=0):
    with torch.no_grad():
        out = model.generate(
            **enc,  # input_ids + attention_mask
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_len = enc["input_ids"].shape[1]
    gen_tokens = out[0][input_len:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

def build_prompt_stage1(persona_instr: str, game_text: str) -> List[Dict[str, str]]:
    user = (
        f"[Persona]\n{persona_instr}\n\n"
        f"[Game]\n{game_text}\n\n"
        "[Task]\nThink step-by-step about what number to request (11-20). "
        "Do NOT output the final number yet. Write a short reasoning note."
    )
    return [{"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":user}]

def build_prompt_stage2(persona_instr: str, game_text: str, thoughts: str) -> List[Dict[str, str]]:
    user = (
        f"[Persona]\n{persona_instr}\n\n"
        f"[Game]\n{game_text}\n\n"
        f"[Previous Thoughts]\n{thoughts}\n\n"
        "[Task]\nBased on your previous thoughts, now give ONLY the final requested amount.\n\n"
        "VERY IMPORTANT RULES:\n"
        "- Output EXACTLY ONE integer between 11 and 20.\n"
        "- Put the number on its own line.\n"
        "- Do NOT write any explanation, reasoning, <think>, words, or anything else before or after the number.\n"
        "- Example of correct output:\n"
        "19\n"
        "- Incorrect examples (do NOT do this):\n"
        "I think 19 is best.\n"
        "<think>19</think>\n"
        "The answer is 19.\n\n"
        "Your output must be exactly like the correct example above."
    )
    return [{"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":user}]

def parse_action_1120(text: str) -> Optional[int]:
    """
    Parse the last valid integer in [11,20].
    If not found, return None (invalid).
    """
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    # try last line first
    if lines:
        m = re.fullmatch(r"(?:\D*)(\d{2})(?:\D*)", lines[-1])
        if m:
            x = int(m.group(1))
            if 11 <= x <= 20:
                return x
    # fallback: find any 11..20
    nums = re.findall(r"\b(1[1-9]|20)\b", text)
    if nums:
        x = int(nums[-1])
        if 11 <= x <= 20:
            return x
    return None

def two_stage_one_shot_action(
    game_text: str,
    persona_instr: str,
    seed: int,
    temperature: float = 1.0,
    top_p: float = 0.95,
    debug: bool = False,
) -> Optional[int]:
    """
    Two-stage elicitation.
    invalid discard: if can't parse final action in [11,20], return None.
    """
    set_global_seed(seed)

    # Stage 1
    m1 = build_prompt_stage1(persona_instr, game_text)
    in1 = apply_chat_template(m1)
    thoughts = _generate_text(in1, max_new_tokens=128, temperature=temperature, top_p=top_p)

    # Stage 2
    m2 = build_prompt_stage2(persona_instr, game_text, thoughts)
    in2 = apply_chat_template(m2)
    final = _generate_text(in2, max_new_tokens=12, temperature=temperature, top_p=top_p)

    if debug:
        print("\n=== DEBUG thoughts ===\n", thoughts)
        print("\n=== DEBUG final raw ===\n", final)

    return parse_action_1120(final)

# -----------------------------
# 5) Population simulation (paper-style)
# -----------------------------
def sample_agent_types(w: np.ndarray, N: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return rng.choice(len(w), size=N, replace=True, p=w)

def empirical_pmf(actions: List[Optional[int]], action_set=ACTION_SET) -> Dict[int, float]:
    valid = [a for a in actions if a in action_set]
    if len(valid) == 0:
        # fallback uniform if everything invalid
        return {a: 1/len(action_set) for a in action_set}
    counts = {a: 0 for a in action_set}
    for a in valid:
        counts[a] += 1
    total = len(valid)
    return {a: counts[a] / total for a in action_set}

def pmf_to_cdf(pmf: Dict[int, float], action_set=ACTION_SET) -> Dict[int, float]:
    cdf, s = {}, 0.0
    for a in action_set:
        s += pmf[a]
        cdf[a] = s
    return cdf

def cdf_L1_distance(pmf_hat: Dict[int,float], pmf_true: Dict[int,float], action_set=ACTION_SET) -> float:
    Fh, Ft = pmf_to_cdf(pmf_hat, action_set), pmf_to_cdf(pmf_true, action_set)
    return float(sum(abs(Fh[a] - Ft[a]) for a in action_set))

def kl_div(pmf_true: Dict[int,float], pmf_hat: Dict[int,float], eps: float = 1e-12) -> float:
    # KL(true || hat)
    s = 0.0
    for a in ACTION_SET:
        p = max(pmf_true[a], eps)
        q = max(pmf_hat[a], eps)
        s += p * math.log(p / q)
    return float(s)

def simulate_population_distribution(
    game_text: str,
    personas: List[Tuple[str,str]],
    w: np.ndarray,
    N_agents: int,
    master_seed: int,
    stage: int,
    game_i: int,
    temperature: float = 1.0,
    top_p: float = 0.95,
    debug_first: bool = False,
) -> Tuple[Dict[int,float], float]:
    """
    Returns:
      pmf_hat over 11..20
      valid_rate
    """
    # sample agent types once
    agent_types = sample_agent_types(w, N_agents, seed=derive_seed(master_seed, stage, game_i, 0, 999))
    actions: List[Optional[int]] = []
    valid_ct = 0

    for ai in range(N_agents):
        pk = int(agent_types[ai])
        _, instr = personas[pk]
        seed = derive_seed(master_seed, stage, game_i, ai, pk)
        a = two_stage_one_shot_action(
            game_text=game_text,
            persona_instr=instr,
            seed=seed,
            temperature=temperature,
            top_p=top_p,
            debug=(debug_first and ai == 0),
        )
        actions.append(a)
        if a is not None:
            valid_ct += 1

    pmf = empirical_pmf(actions)
    valid_rate = valid_ct / N_agents
    return pmf, valid_rate

def simulate_baseline_distribution(
    game_text: str,
    N_agents: int,
    master_seed: int,
    stage: int,
    game_i: int,
    temperature: float = 1.0,
    top_p: float = 0.95,
) -> Tuple[Dict[int,float], float]:
    _, instr = BASELINE_PROMPT
    actions: List[Optional[int]] = []
    valid_ct = 0
    for ai in range(N_agents):
        seed = derive_seed(master_seed, stage, game_i, ai, 0)
        a = two_stage_one_shot_action(game_text, instr, seed=seed, temperature=temperature, top_p=top_p)
        actions.append(a)
        if a is not None:
            valid_ct += 1
    pmf = empirical_pmf(actions)
    return pmf, valid_ct / N_agents

# -----------------------------
# 6) Fit mixture weights by minimizing CDF distance on BASIC
# -----------------------------
def project_to_simplex(v: np.ndarray) -> np.ndarray:
    v = v.astype(float)
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    rho = np.where(u - (cssv - 1) / (np.arange(len(u)) + 1) > 0)[0]
    if len(rho) == 0:
        return np.ones_like(v) / len(v)
    rho = rho[-1]
    theta = (cssv[rho] - 1) / (rho + 1)
    w = np.maximum(v - theta, 0)
    return w / (w.sum() + 1e-12)

def fit_weights_cdf_distance(
    personas: List[Tuple[str,str]],
    human_pmf_basic: Dict[int,float],
    N_agents_fit: int,
    master_seed: int,
    iters: int = 12,
    lr: float = 0.6,
    fd_eps: float = 0.03,
    temperature: float = 1.0,
    top_p: float = 0.95,
    verbose: bool = True,
) -> np.ndarray:
    """
    Paper-style: objective uses population simulation => nondiff => finite-diff gradient.
    Loss = CDF-L1 distance between simulated mixture PMF and human PMF on BASIC.
    """
    K = len(personas)
    w = np.ones(K) / K

    def eval_loss(w_eval: np.ndarray) -> Tuple[float, float, Dict[int,float]]:
        pmf_hat, valid_rate = simulate_population_distribution(
            game_text=build_1120_game_text("basic"),
            personas=personas,
            w=w_eval,
            N_agents=N_agents_fit,
            master_seed=master_seed,
            stage=0,
            game_i=0,
            temperature=temperature,
            top_p=top_p,
            debug_first=False,
        )
        loss = cdf_L1_distance(pmf_hat, human_pmf_basic)
        return loss, valid_rate, pmf_hat

    loss0, vr0, _ = eval_loss(w)
    if verbose:
        print(f"Initial BASIC loss (CDF-L1) = {loss0:.4f} | valid_rate={vr0:.4f}")

    base_loss = loss0
    for it in range(iters):
        grad = np.zeros(K)
        for k in range(K):
            w_plus = w.copy()
            w_plus[k] += fd_eps
            w_plus = project_to_simplex(w_plus)
            L_plus, _, _ = eval_loss(w_plus)
            grad[k] = (L_plus - base_loss) / fd_eps

        w_new = project_to_simplex(w - lr * grad)
        new_loss, vr, _ = eval_loss(w_new)

        if verbose:
            print(f"Iter {it:02d}: loss={new_loss:.4f} | valid_rate={vr:.4f} | w={np.round(w_new,3)}")

        if abs(base_loss - new_loss) < 0.001:  # 更嚴格
            w = w_new
            base_loss = new_loss
            break

        w, base_loss = w_new, new_loss

    if verbose:
        print("\nEstimated weights:")
        for (name, _), ww in zip(personas, w):
            print(f"  {name:18s}: {ww:.4f}")
    return w

# -----------------------------
# 7) Full evaluation: BASIC fit + validation on COSTLESS/CYCLE + negative controls
# -----------------------------
def eval_family(
    family_name: str,
    personas: List[Tuple[str,str]],
    master_seed: int,
    N_fit: int = 200,          # 加大到 200（更穩定，接近論文規模）
    N_eval: int = 500,         # 加大到 500（KL 更可靠）
    temperature: float = 0.8,  # 略降，避免過度隨機
    top_p: float = 0.92,
):
    print("\n" + "="*100)
    print(f"FAMILY: {family_name} | master_seed={master_seed} | N_fit={N_fit} N_eval={N_eval}")
    print("="*100)

    # Fit on BASIC
    w_hat = fit_weights_cdf_distance(
        personas=personas,
        human_pmf_basic=HUMAN_DIST["basic"],
        N_agents_fit=N_fit,
        master_seed=master_seed,
        iters=30,               # 略減，加速
        lr=0.6,
        fd_eps=0.03,
        temperature=temperature,
        top_p=top_p,
        verbose=True,
    )

    # Evaluate on BASIC + COSTLESS + CYCLE using KL (like paper figures)
    for gi, variant in enumerate(["basic", "costless", "cycle"]):
        game_text = build_1120_game_text(variant)
        pmf_mix, vr_mix = simulate_population_distribution(
            game_text=game_text,
            personas=personas,
            w=w_hat,
            N_agents=N_eval,
            master_seed=master_seed,
            stage=gi,      # 0/1/2
            game_i=gi,
            temperature=temperature,
            top_p=top_p,
            debug_first=(variant=="costless"),
        )
        pmf_base, vr_base = simulate_baseline_distribution(
            game_text=game_text,
            N_agents=N_eval,
            master_seed=master_seed,
            stage=3,
            game_i=gi,
            temperature=temperature,
            top_p=top_p,
        )

        kl_mix = kl_div(HUMAN_DIST[variant], pmf_mix)
        kl_base = kl_div(HUMAN_DIST[variant], pmf_base)

        print(f"\n[{variant.upper()}] valid_rate mix={vr_mix:.3f} base={vr_base:.3f}")
        print(f"KL(human || mix)  = {kl_mix:.4f}")
        print(f"KL(human || base) = {kl_base:.4f}")
        if kl_base > 0:
            print(f"Relative KL reduction vs baseline = {(1 - kl_mix/kl_base)*100:.1f}%")

        # quick diagnostic: where the mass is (to see 'collapse to 19' patterns)
        top_mix = sorted(pmf_mix.items(), key=lambda x: -x[1])[:3]
        top_base = sorted(pmf_base.items(), key=lambda x: -x[1])[:3]
        print("Top-3 mix:", top_mix)
        print("Top-3 base:", top_base)

    return w_hat

def run_all(master_seed: int = 7):
    # Strategic (theoretical)
    w_strat = eval_family("STRATEGIC_REASONING", STRATEGIC_PROMPTS, master_seed)

    # Negative controls
    w_mbti = eval_family("MBTI", MBTI_PROMPTS, master_seed)
    w_hist = eval_family("HISTORICAL_FIGURES", HIST_PROMPTS, master_seed)
    w_always = eval_family("ALWAYS_PICK_N", ALWAYS_PICK_PROMPTS, master_seed)

    return {
        "w_strat": w_strat,
        "w_mbti": w_mbti,
        "w_hist": w_hist,
        "w_always": w_always,
    }

# -----------------------------
# 8) Robustness over master seeds
# -----------------------------
def robustness(seeds=(7, 11, 97), N_fit=200, N_eval=800):
    results = []
    for s in seeds:
        out = run_all(master_seed=s)
        results.append((s, out))
    return results

# -----------------------------
# MAIN (example)
# -----------------------------
# out = run_all(master_seed=7)
# rob = robustness(seeds=(7,11), N_fit=200, N_eval=800)

In [ ]:
w_strat_quick = eval_family(
    family_name="STRATEGIC_QUICK",
    personas=STRATEGIC_PROMPTS,
    master_seed=7,
    N_fit=2,          # 減到 10，計算量減半
    N_eval=1,
    temperature=0.75,
    top_p=0.90,
)
print("Strategic weights:", w_strat_quick.round(3))


FAMILY: STRATEGIC_QUICK | master_seed=7 | N_fit=2 N_eval=1
Initial BASIC loss (CDF-L1) = 2.0200 | valid_rate=1.0000


In [ ]:
out = run_all(master_seed=7)


FAMILY: STRATEGIC_REASONING | master_seed=7 | N_fit=60 N_eval=300
